# VaidyaLM: QLoRA fine-tuning on Kaggle
Settings: Accelerator **GPU T4**, Internet **On**. Add-ons → Secrets: add `HF_TOKEN` (write scope).

Edit `HF_USER` below, then Run All. One config ≈ 1.5–3 h on a T4 with `max_train_rows=10000`.

In [ ]:
HF_USER = 'your-hf-username'                 # <- change
DATASET = f'{HF_USER}/vaidyalm-instruct'      # pushed by the preprocess_corpus DAG
CONFIG = 'r16'                                # r8 | r16 | r32 | r16_medquad | r16_medquad_artificial
import os
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['CUDA_VISIBLE_DEVICES'] = '0'      # one GPU: 4-bit models and multi-GPU DataParallel don't mix

In [ ]:
!git clone https://github.com/aad1tyaaaaa/VaidyaLM.git
%cd VaidyaLM
!git pull

In [ ]:
!pip install -q -U 'transformers>=4.56' peft accelerate bitsandbytes datasets scikit-learn pyyaml

Smoke test: 200 rows. If this runs out of memory, set `batch_size: 2` + `grad_accum: 8` in the config.

In [ ]:
!python -m src.train.sft_lora src/train/configs/{CONFIG}.yaml --dataset {DATASET} --max-train-rows 200 --output-root /kaggle/working/smoke

Full run. Checkpoints are pushed to the Hub every `save_steps`. If the session dies, just rerun this cell: it resumes.

In [ ]:
!python -m src.train.sft_lora src/train/configs/{CONFIG}.yaml --dataset {DATASET} --hub-user {HF_USER} --output-root /kaggle/working/outputs

## Loss curve

In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open(f'results/train_logs/{CONFIG}.json'))['log_history']
tr = [(e['step'], e['loss']) for e in h if 'loss' in e]; ev = [(e['step'], e['eval_loss']) for e in h if 'eval_loss' in e]
plt.plot(*zip(*tr), label='train'); plt.plot(*zip(*ev), 'o-', label='dev'); plt.legend(); plt.xlabel('step'); plt.ylabel('loss')

## Evaluate the adapter (same script as the baseline)
Dev set first (Phase 4 exit criterion), then the test set with all 3 prompts.

In [ ]:
ADAPTER = f'{HF_USER}/vaidyalm-qwen1.5b-lora-{CONFIG}'
!python -m src.eval.infer --prompt structured --dataset dev --dev-source {DATASET} --out results/dev.csv
!python -m src.eval.infer --prompt structured --dataset dev --dev-source {DATASET} --adapter {ADAPTER} --out results/dev.csv
import pandas as pd; pd.read_csv('results/dev.csv')

In [ ]:
!python -m src.eval.infer --prompt zero_shot  --adapter {ADAPTER} --out results/finetuned.csv
!python -m src.eval.infer --prompt few_shot   --adapter {ADAPTER} --batch-size 4 --out results/finetuned.csv
!python -m src.eval.infer --prompt structured --adapter {ADAPTER} --out results/finetuned.csv
pd.read_csv('results/finetuned.csv')

Download from the Output panel: `results/train_logs/`, `results/*.csv` and the `results/predictions/` folders. Copy `predictions/` into your local repo's `results/predictions/` so the `benchmark_models` DAG reuses them.

In [ ]:
!cd /kaggle/working/VaidyaLM && zip -qr /kaggle/working/results.zip results && ls -lh /kaggle/working/results.zip